In [1]:
import os
import glob
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BATCH_SIZE = 32
IMG_SIZE = (224, 224)
MAX_SAMPLES_PER_AGE = 400
EPOCHS = 20

DATA_DIR_CANDIDATES = [
    r"C:\Users\savan\Desktop\CS\Deep learning\exercise3\UTKFace",
    "/mnt/c/Users/savan/Desktop/CS/Deep learning/exercise3/UTKFace",
    "exercise3/UTKFace",
]

DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if os.path.isdir(path)), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find UTKFace dataset. Check DATA_DIR_CANDIDATES.")

print(f"Using dataset: {DATA_DIR}")


Using dataset: C:\Users\savan\Desktop\CS\Deep learning\exercise3\UTKFace


In [2]:
def prepare_dataframe(data_dir):
    image_paths = []
    for extension in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        image_paths.extend(glob.glob(os.path.join(data_dir, extension)))

    if not image_paths:
        raise ValueError(f"No images found in {data_dir}")

    rows = []
    skipped = 0
    for image_path in image_paths:
        filename = os.path.basename(image_path)
        try:
            age = int(filename.split("_")[0])
        except (ValueError, IndexError):
            skipped += 1
            continue

        if 0 <= age <= 120:
            rows.append({"filepath": image_path, "age": age})
        else:
            skipped += 1

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("No valid UTKFace filenames were found.")

    df = (
        df.sample(frac=1, random_state=SEED)
          .groupby("age", group_keys=False)
          .head(MAX_SAMPLES_PER_AGE)
          .sample(frac=1, random_state=SEED)
          .reset_index(drop=True)
    )

    print(f"Valid images: {len(df)}")
    print(f"Skipped files: {skipped}")
    print(f"Age range: {df['age'].min()} to {df['age'].max()}")
    return df

all_df = prepare_dataframe(DATA_DIR)
train_df, val_df = train_test_split(
    all_df,
    test_size=0.2,
    random_state=SEED,
    stratify=pd.cut(all_df["age"], bins=10, labels=False, duplicates="drop"),
)

print(f"Train images: {len(train_df)}")
print(f"Validation images: {len(val_df)}")
train_df.head()


Valid images: 25730
Skipped files: 0
Age range: 1 to 116
Train images: 20584
Validation images: 5146


,filepath,age
23027,C:\Users\savan\Desktop\CS\Deep learning\exerci...,5
10373,C:\Users\savan\Desktop\CS\Deep learning\exerci...,34
13768,C:\Users\savan\Desktop\CS\Deep learning\exerci...,27
15905,C:\Users\savan\Desktop\CS\Deep learning\exerci...,22
11027,C:\Users\savan\Desktop\CS\Deep learning\exerci...,42


In [ ]:
def process_path(filepath, age):
    image_bytes = tf.io.read_file(filepath)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMG_SIZE)
    image = preprocess_input(tf.cast(image, tf.float32))
    return image, tf.cast(age, tf.float32)


def create_dataset(dataframe, shuffle=False):
    filepaths = dataframe["filepath"].astype(str).values
    ages = dataframe["age"].astype("float32").values

    dataset = tf.data.Dataset.from_tensor_slices((filepaths, ages))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(dataframe), seed=SEED, reshuffle_each_iteration=True)

    return (
        dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
               .batch(BATCH_SIZE)
               .prefetch(tf.data.AUTOTUNE)
    )

train_dataset = create_dataset(train_df, shuffle=True)
val_dataset = create_dataset(val_df)

sample_images, sample_ages = next(iter(train_dataset))
print(sample_images.shape, sample_ages.shape)


In [ ]:
def build_model():
    try:
        base_model = MobileNetV2(
            weights="imagenet",
            include_top=False,
            input_shape=(*IMG_SIZE, 3),
        )
    except Exception as error:
        print(f"Could not load ImageNet weights ({error}). Training MobileNetV2 from scratch.")
        base_model = MobileNetV2(
            weights=None,
            include_top=False,
            input_shape=(*IMG_SIZE, 3),
        )

    base_model.trainable = False

    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation="linear", name="age")(x)

    model = keras.Model(inputs, outputs, name="mobilenetv2_age_estimator")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss="mae",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model

model = build_model()
model.summary()


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_mae",
        patience=5,
        restore_best_weights=True,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        "best_age_estimation_mobilenet.keras",
        monitor="val_mae",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
]

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
)


In [ ]:
def plot_history(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history.history["mae"], label="Training MAE")
    plt.plot(history.history["val_mae"], label="Validation MAE")
    plt.title("MobileNetV2 Age Estimation")
    plt.xlabel("Epoch")
    plt.ylabel("MAE (years)")
    plt.grid(True)
    plt.legend()
    plt.show()

plot_history(history)


In [ ]:
val_loss, val_mae = model.evaluate(val_dataset)
print(f"Validation MAE: {val_mae:.2f} years")

for images, true_ages in val_dataset.take(1):
    predicted_ages = model.predict(images[:8]).reshape(-1)
    for true_age, predicted_age in zip(true_ages[:8].numpy(), predicted_ages):
        print(f"True: {true_age:5.1f} | Predicted: {predicted_age:5.1f}")
